# ViSceT5 + TWC — Finetune / Predict ở commit tuỳ chọn · Colab

Notebook clone **repo GitHub `GIT_REPO`** và chuyển code sang đúng **`GIT_COMMIT`** (mã commit, tag hoặc tên nhánh), rồi train / predict bằng chính code đó: `setup.sh` → `run_pipeline.py` → `training/finetune.py configs/finetune.yaml` → push checkpoint mỗi epoch lên HF. Thông số không có trong cell 2 lấy từ `configs/finetune.yaml` **của commit đó**.

Mặc định: commit **`4221b1f`** và cấu hình của run `Kus669/ViSceT5-twcbr-vitextvqa-consformer-twc-p35-tau0.2-bs4x2` (`checkpoint-21975`: dev 30.32 / 56.68, test 31.64 / 56.09). Đã so `training_args.bin` + `config.json` của run đó với cấu hình này: không khác tham số nào.

| Biến (cell 2) | Ý nghĩa |
|---|---|
| `GIT_REPO`, `GIT_COMMIT` | repo GitHub và commit / tag / nhánh của code dùng để train và predict |
| `QAVIT` | `"off"`: CLIP thường, không adapter · `"no_question"`: adapter QA-ViT ở 6 lớp cuối CLIP được học, câu hỏi rỗng (chỉ có ở commit đọc `QAVIT_MODE`) · `"question"`: QA-ViT đầy đủ |
| `USE_CONSFORMER` | `True`: OCR qua Consformer (ConsFormer + ký tự + box + det/rec) · `False`: OCR text-only |
| `USE_TWC` | `True`: thêm bản OCR related (OCR Augmentation) và loss TWC · `False`: chỉ OCR gốc |
| `TWC_AUG_P` | xác suất một từ của bản related bị đổi |
| `TWC_TAU` | temperature của TWC (học được hay cố định tuỳ code của commit) |
| `BATCH_SIZE`, `GRAD_ACCUM` | batch mỗi bước và số bước tích luỹ gradient (batch hiệu dụng = tích) |
| `DATASET`, `PREDICT_CKPT` | dataset và checkpoint dùng ở mục 6 / 8 |

Cell 3 đọc `run_pipeline.py` của commit để biết nó nhận những biến môi trường nào, chỉ đặt các biến đó và **dừng** nếu một công tắc không áp dụng được ở commit này (ví dụ `QAVIT="no_question"` ở commit chưa có chế độ đó, hoặc batch khác yaml ở commit chưa đọc biến batch).
Tên run gồm mã commit: thư mục `output/<commit>_<DATASET>_<RUN_NAME>` và repo HF `ViSceT5-<commit>-<dataset>-<RUN_NAME>`, nên run ở commit khác nhau không bao giờ resume nhầm của nhau.

Ví dụ ablation chạy song song (mỗi run một phiên Colab, cùng `GIT_COMMIT = "4221b1f"`): `USE_CONSFORMER=False, USE_TWC=True` (chỉ TWC) và `USE_CONSFORMER=False, USE_TWC=False` (baseline: không QA-ViT, không Consformer, không TWC).

## 1. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 2. Cấu hình — code (repo + commit) và module

In [ ]:
GIT_REPO       = "https://github.com/Kussssssss/ViSceT5.git"
GIT_COMMIT     = "4221b1f"
QAVIT          = "off"
USE_CONSFORMER = True
USE_TWC        = True
TWC_AUG_P      = 0.35
TWC_TAU        = 0.2
BATCH_SIZE     = 4
GRAD_ACCUM     = 2
DATASET        = "ViTextVQA"
PREDICT_CKPT   = "best"

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert QAVIT in ("off", "no_question", "question"), "QAVIT: off | no_question | question"
assert 0 <= TWC_AUG_P <= 1 and TWC_TAU > 0 and BATCH_SIZE >= 1 and GRAD_ACCUM >= 1
RUN_NAME = "-".join([m for m, on in (("qavit", QAVIT == "question"), ("qavitnq", QAVIT == "no_question"),
                                     ("consformer", USE_CONSFORMER),
                                     (f"twc-p{int(round(100 * TWC_AUG_P))}-tau{TWC_TAU:g}", USE_TWC)) if on] or ["base"])
RUN_NAME += f"-bs{BATCH_SIZE}x{GRAD_ACCUM}"
print(f"Code: {GIT_REPO} @ {GIT_COMMIT}")
print(f"Cấu hình: QA-ViT={QAVIT} | Consformer={USE_CONSFORMER} | TWC={USE_TWC}"
      f"{f' (p={TWC_AUG_P}, τ={TWC_TAU})' if USE_TWC else ''} | batch={BATCH_SIZE}x{GRAD_ACCUM} | RUN_NAME={RUN_NAME}")

## 3. Lấy code đúng commit và đặt biến môi trường
Clone `GIT_REPO` (lần đầu), chuyển sang `GIT_COMMIT`, chạy `setup.sh` của commit đó, rồi đặt biến môi trường theo những gì `run_pipeline.py` của commit đó đọc được. Chạy lại cell này mỗi khi đổi cell 2.

In [ ]:
import os, re, subprocess, yaml
REPO_DIR = "/content/ViSceT5"
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q {GIT_REPO} {REPO_DIR}
%cd {REPO_DIR}
!git remote set-url origin {GIT_REPO}
!git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags


def _resolve(ref):
    for cand in (f"origin/{ref}", ref):
        out = subprocess.run(["git", "rev-parse", "--verify", "-q", cand + "^{commit}"], capture_output=True, text=True)
        if out.returncode == 0:
            return out.stdout.strip()
    raise ValueError(f"Không tìm thấy commit/nhánh/tag '{ref}' trong {GIT_REPO}")


CODE_SHA = _resolve(GIT_COMMIT)
CODE_TAG = CODE_SHA[:7]
subprocess.run(["git", "checkout", "-q", "--detach", CODE_SHA], check=True)
!git log --oneline -1
!bash setup.sh

READ = set(re.findall(r'"([A-Z][A-Z_]+)": "[a-z_]+"', open("run_pipeline.py", encoding="utf-8").read()))
YAML_CFG = yaml.safe_load(open("configs/finetune.yaml", encoding="utf-8"))
for _name in ("ABLATION_USE_IMAGE", "ABLATION_USE_OCR", "ABLATION_USE_OCR_AUG", "ABLATION_USE_QACLIP", "CLIP_LATE_ADAPTER",
              "DATASET_NAME", "GRADIENT_ACCUMULATION_STEPS", "HF_REPO", "LEARNING_RATE", "MOCK_TEST", "MODEL_NAME_OR_PATH",
              "NEW_MODULE_LR", "NUM_TRAIN_EPOCHS", "OCR_AUG_MODE", "OCR_FEATURE_NORM", "OCR_RELATED", "OCR_LATTICE_KEEP",
              "OUTPUT_DIR", "PER_DEVICE_EVAL_BATCH_SIZE", "PER_DEVICE_TRAIN_BATCH_SIZE", "QAVIT_MODE",
              "RESUME_CHECKPOINT_ID", "RESUME_FROM_CHECKPOINT", "SCALAR_LR", "STAGE", "TRAIN_CONFIG",
              "TWC_ADV_PROBABILITY", "TWC_ANSWER_WEIGHT", "TWC_FEATURE", "TWC_LOSS", "TWC_MASK_SAME_TEXT", "TWC_OBJECTIVE",
              "TWC_QUEUE_SIZE", "TWC_QUEUE_START", "TWC_RELATED_TRAIN", "TWC_TEMPERATURE", "TWC_VIEW", "TWC_WEIGHT",
              "USE_TWC", "GIT_BRANCH", "GIT_COMMIT", "PREDICT_CKPT_DIR", "PREDICT_DATASET", "PREDICT_SPLIT"):
    os.environ.pop(_name, None)

_b = lambda v: "true" if v else "false"
OUTPUT_DIR = f"/content/ViSceT5/output/{CODE_TAG}_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-{CODE_TAG}-{DATASET.lower()}-{RUN_NAME}"
ENV = {"DATASET_NAME": DATASET, "ABLATION_USE_OCR": _b(USE_CONSFORMER), "USE_TWC": _b(USE_TWC),
       "TWC_ADV_PROBABILITY": str(TWC_AUG_P), "TWC_TEMPERATURE": str(TWC_TAU)}
_missing = [k for k in ENV if k not in READ]
assert not _missing, f"Commit {CODE_TAG}: run_pipeline.py không đọc {_missing} → không áp dụng được cấu hình ở cell 2."
if "QAVIT_MODE" in READ:
    ENV["QAVIT_MODE"] = QAVIT
else:
    assert QAVIT != "no_question", f"Commit {CODE_TAG} chưa có QA-ViT no_question: đặt QAVIT = \"off\" hoặc \"question\"."
if "ABLATION_USE_QACLIP" in READ:
    ENV["ABLATION_USE_QACLIP"] = _b(QAVIT == "question")
if "ABLATION_USE_OCR_AUG" in READ:
    ENV["ABLATION_USE_OCR_AUG"] = _b(USE_TWC)
for _name, _key, _value in (("PER_DEVICE_TRAIN_BATCH_SIZE", "per_device_train_batch_size", BATCH_SIZE),
                            ("GRADIENT_ACCUMULATION_STEPS", "gradient_accumulation_steps", GRAD_ACCUM)):
    if _name in READ:
        ENV[_name] = str(_value)
    else:
        assert YAML_CFG.get(_key) == _value, (f"Commit {CODE_TAG} không đọc {_name}: batch lấy từ yaml "
                                              f"({_key} = {YAML_CFG.get(_key)}) → đặt BATCH_SIZE / GRAD_ACCUM đúng giá trị đó.")
os.environ.update(ENV, OUTPUT_DIR=OUTPUT_DIR, HF_REPO=HF_REPO, PYTHONUNBUFFERED="1")
print(f"Code {CODE_TAG} | biến môi trường: {ENV}")
print(f"OUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}\nPREDICT_CKPT={PREDICT_CKPT}")

## 4. (Tuỳ chọn) Mock test — vài bước để bắt lỗi trước khi chạy thật
`--smoke_test`: 8 mẫu train, 4 mẫu val, 3 bước; không push, không resume từ HF.

In [ ]:
!git checkout -q --detach {CODE_SHA}
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock MOCK_TEST=true STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5. Finetune (từ đầu, không pretrain) — log train có `twc_loss`, `twc_acc`, `twc_tau` khi bật TWC

In [ ]:
!git checkout -q --detach {CODE_SHA}
!MOCK_TEST=false STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5b. (Khi cần) Đẩy checkpoint local lên HF rồi train tiếp
Dùng khi đã train mà **chưa có `HF_TOKEN`** (checkpoint chỉ nằm trên Colab): chạy cell 1 (token) → cell 2 (cấu hình, đúng commit và công tắc của lần train) → cell 3 → cell này. Cell đẩy các `checkpoint-N` trong `OUTPUT_DIR` (kèm optimizer/scheduler để resume) lên `HF_REPO`, sau đó chạy lại **cell 5**: `finetune.py` tự resume từ checkpoint mới nhất trên HF và từ đó push mỗi epoch. Chỉ checkpoint đã lưu cuối epoch mới có; phần đang train dở của epoch sau bị mất. Nếu runtime Colab đã reset thì không còn gì để đẩy.

In [ ]:
!ls -d {OUTPUT_DIR}/checkpoint-* 2>/dev/null
!python scripts/hf_io.py upload_checkpoints --repo {HF_REPO} --dir {OUTPUT_DIR}

## 6. Dự đoán dev + test (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Test được chấm bằng `training/benchmark_eval.py` — **đúng giao thức benchmark** (cùng cách chấm các dòng baseline/LMM trong bài): EM, F1 token/char, BLEU-1..4/avg, CIDEr, BERTScore (mBERT).
Đáp án test lấy từ `eval_gt/<DATASET>_test_gt.json` trong repo (đã có `ViTextVQA`); dataset chưa có file thì cell hiện nút upload **ngay đầu cell** (trước khi predict).
Nếu `OUTPUT_DIR` (hoặc `predictions/` trên HF) đã có `submission_test.csv` thì cell **bỏ qua predict** và chỉ chấm điểm.
File nộp, log predict (có EM/F1 dev) và kết quả chấm test được upload vào thư mục `predictions/` của repo HF của cấu hình.

**Chọn checkpoint:** `PREDICT_CKPT` ở cell 2. Cell in bảng các checkpoint kèm EM/F1/CIDEr trên val của từng epoch rồi dùng checkpoint được chọn:
`best` = model tốt nhất (thư mục gốc, do `trainer.save_model` lưu sau `load_best_model_at_end`), `latest`, `epoch3`, hoặc `checkpoint-17580`.
Chỉ tải file inference của checkpoint đó (không tải optimizer). Kết quả của `best` nằm ở `OUTPUT_DIR` và `predictions/` trên HF;
của checkpoint khác nằm ở `OUTPUT_DIR/predict_checkpoint-N` và `predictions/checkpoint-N/` → không đè nhau.

**Predict lại sau khi Colab đã tắt:** chạy cell 1, 2, 3 (đặt đúng commit, công tắc của lần train và `PREDICT_CKPT`) rồi chạy thẳng cell này, bỏ qua mock và finetune.

In [ ]:
import os, shutil, glob
!git checkout -q --detach {CODE_SHA}
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    from google.colab import files
    print(f"⬆️ Chọn file {os.path.basename(TEST_GT_JSON)} để upload (bấm Cancel nếu chưa có → vẫn predict, chưa chấm test)")
    up = files.upload()
    if up:
        shutil.move(os.path.abspath(next(iter(up))), TEST_GT_JSON)
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có")

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, PRED_DIR, PRED_REPO_PATH = CKPT_INFO["model_dir"], CKPT_INFO["pred_dir"], CKPT_INFO["pred_path_in_repo"]
assert os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")), f"Không có model trong {MODEL_DIR}"

tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
pred = os.path.join(PRED_DIR, f"submission_{tag}test.csv")
if os.path.exists(pred):
    print(f"✅ Đã có {pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{tag}{sp}.csv")
    !MOCK_TEST=false STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_CKPT_DIR={MODEL_DIR} HF_REPO= python3 run_pipeline.py 2>&1 | tee train_execution.log
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(PRED_DIR, "predict_log.txt"))
log = os.path.join(PRED_DIR, "predict_log.txt")
dev_lines = [l.strip() for l in open(log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(log) else []
if dev_lines:
    for line in dev_lines[-2:]:
        print(line)
if os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
    !pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {pred} --out {PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa đặt TEST_GT_JSON → chưa chấm test.")

if os.environ.get("HF_TOKEN") and HF_REPO:
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {PRED_DIR} --path_in_repo {PRED_REPO_PATH} --files submission_{tag}dev.csv submission_{tag}test.csv predict_log.txt eval_test.json
else:
    print("⚠️ Không có HF_TOKEN/HF_REPO → file dự đoán chỉ nằm trên Colab, hãy tải về trước khi ngắt phiên.")

## 6b. Dự đoán + chấm điểm một repo HF bất kỳ
Chọn thẳng repo model trên Hugging Face (`HF_PRED_REPO`) và checkpoint (`HF_PRED_CKPT`: `best` | `latest` | `epochN` | `checkpoint-N`), không phụ thuộc cấu hình ở cell 2. Các bước giống mục 6; kết quả upload vào `predictions/` (hoặc `predictions/checkpoint-N/`) của chính repo đó. Repo đã có file dự đoán của checkpoint đó thì chỉ chấm lại.

**`HF_PRED_COMMIT`:** commit code dùng để predict — phải là code đã train run đó, để model dựng được và bản OCR related được tạo đúng như lúc train. `"auto"` (mặc định):

| repo / config của checkpoint | commit |
|---|---|
| tên repo có mã commit (`ViSceT5-<commit>-…`, run của notebook này) | commit đó |
| `ablation_use_ocr_aug=true`, `ocr_aug_mode="separate"` (nhánh `tmp`, ví dụ `Kus669/tmp_viscet5`) | `2090b2d` |
| `ablation_use_ocr_aug=true`, lattice / concat (ví dụ lat-p70) | `bf74388` |
| TWC bật, chưa có trường `clip_late_adapter` (run `…-consformer-twc-…`: bs8x2, bs4x2) | `4221b1f` (có trường box/layout: `5545dce` / `1046e86`) |
| còn lại | code của cell 3 |

Sau khi predict, code quay lại commit của cell 3.

In [ ]:
HF_PRED_REPO    = "Kus669/ViSceT5-twcbr-vitextvqa-consformer-twc-p35-tau0.2-bs4x2"
HF_PRED_CKPT    = "checkpoint-21975"
HF_PRED_COMMIT  = "auto"
HF_PRED_DATASET = DATASET

import os, re, shutil, json, subprocess
!git checkout -q --detach {CODE_SHA}
HF_PRED_DIR = f"/content/ViSceT5/output/hf_{HF_PRED_REPO.split('/')[-1]}"
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{HF_PRED_DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{HF_PRED_DATASET}_test_gt.json"
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có (chạy mục 6 một lần để upload)")

if os.path.exists("/content/hf_ckpt_info.json"):
    os.remove("/content/hf_ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_PRED_REPO} --dir {HF_PRED_DIR} --ckpt {HF_PRED_CKPT} --info_out /content/hf_ckpt_info.json
HF_INFO = json.load(open("/content/hf_ckpt_info.json", encoding="utf-8"))
H_MODEL_DIR, H_PRED_DIR, H_REPO_PATH = HF_INFO["model_dir"], HF_INFO["pred_dir"], HF_INFO["pred_path_in_repo"]

h_tag = "" if HF_PRED_DATASET == "ViTextVQA" else f"{HF_PRED_DATASET}_"
h_pred = os.path.join(H_PRED_DIR, f"submission_{h_tag}test.csv")
h_cfg = json.load(open(os.path.join(H_MODEL_DIR, "config.json"), encoding="utf-8"))
h_commit = HF_PRED_COMMIT
h_named = re.match(r"ViSceT5-([0-9a-f]{7})-", HF_PRED_REPO.split("/")[-1])
if h_commit == "auto":
    if h_named:
        h_commit = h_named.group(1)
    elif h_cfg.get("ablation_use_ocr_aug"):
        h_commit = "2090b2d" if h_cfg.get("ocr_aug_mode") == "separate" else "bf74388"
    elif h_cfg.get("use_twc") and h_cfg.get("twc_objective") in ("answer", "both"):
        h_commit = "bf74388"
    elif h_cfg.get("use_twc") and "clip_late_adapter" not in h_cfg:
        h_commit = ("1046e86" if "ocr_related_remove_layout" in h_cfg
                    else "5545dce" if "ocr_related_remove_bbox" in h_cfg else "4221b1f")
    else:
        h_commit = ""
h_sha = _resolve(h_commit) if h_commit else CODE_SHA
print(f"Code dùng để predict: {h_sha[:7]}")
if os.path.exists(h_pred):
    print(f"✅ Đã có {h_pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{h_tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{h_tag}{sp}.csv")
    !git checkout -q --detach {h_sha}
    !MOCK_TEST=false STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={HF_PRED_DATASET} PREDICT_CKPT_DIR={H_MODEL_DIR} HF_REPO= python3 run_pipeline.py 2>&1 | tee train_execution.log
    !git checkout -q --detach {CODE_SHA}
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{h_tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(H_PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(H_PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(H_PRED_DIR, "predict_log.txt"))
h_log = os.path.join(H_PRED_DIR, "predict_log.txt")
h_dev = [l.strip() for l in open(h_log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(h_log) else []
for line in h_dev[-2:]:
    print(line)
if os.path.exists(TEST_GT_JSON) and os.path.exists(h_pred):
    !pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {h_pred} --out {H_PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa có GT test hoặc file dự đoán → chưa chấm test.")

if os.environ.get("HF_TOKEN") and os.path.exists(h_pred):
    !python scripts/hf_io.py upload_predictions --repo {HF_PRED_REPO} --dir {H_PRED_DIR} --path_in_repo {H_REPO_PATH} --files submission_{h_tag}dev.csv submission_{h_tag}test.csv predict_log.txt eval_test.json

## 7. Tổng hợp kết quả mọi cấu hình đã chạy

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for od in sorted(glob.glob(f"/content/ViSceT5/output/*_{DATASET}_*")):
    if not os.path.isdir(od) or od.endswith("_mock"):
        continue
    tag, name = os.path.basename(od).split(f"_{DATASET}_", 1)
    mods = set(name.split("-"))
    r = {"code": tag, "run": name, "QA-ViT": "question" if "qavit" in mods else ("no_question" if "qavitnq" in mods else "off"),
         "Consformer": "consformer" in mods or "scespavis" in mods, "TWC": "twc" in mods}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        hist = json.load(open(st[-1])).get("log_history", [])
        key = "eval_cider" if DATASET == "OpenViVQA" else "eval_em_f1"
        evs = [h for h in hist if "eval_em" in h]
        if evs:
            best = max(evs, key=lambda h: h.get(key, 0.5 * (h.get("eval_em", 0) + h.get("eval_f1", 0))))
            r.update({"val_EM": round(100 * best.get("eval_em", float("nan")), 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    for pd_dir in [od] + sorted(glob.glob(os.path.join(od, "predict_checkpoint-*"))):
        rr = dict(r, ckpt="best" if pd_dir == od else os.path.basename(pd_dir)[len("predict_"):])
        ev = os.path.join(pd_dir, "eval_test.json")
        if os.path.exists(ev):
            t = json.load(open(ev, encoding="utf-8"))
            rr.update({"test_EM": round(100 * t.get("Exact Match (EM)", float("nan")), 2), "test_F1": round(100 * t.get("F1 (Token-level)", float("nan")), 2),
                       "test_BLEU4": round(t.get("BLEU-4", float("nan")), 4), "test_CIDEr": round(t.get("CIDEr", float("nan")), 4),
                       "test_BERTScore": round(100 * t.get("BERTScore_F1", float("nan")), 2)})
        elif pd_dir != od:
            continue
        rows.append(rr)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")

## 8. Trực quan hoá attention (cấu hình hiện tại)
`scripts/visualize_attention.py` sinh đáp án (beam search như predict) rồi lấy **cross-attention** của các token đáp án lên đầu vào encoder,
trung bình trên mọi lớp và mọi head. Mỗi câu hỏi ra một hình: heatmap trên 14×14 patch ảnh (nếu mô hình dùng ảnh), box OCR tô màu theo attention,
và attention trên từng từ của câu hỏi, OCR gốc, OCR related (viền đỏ = từ bị đổi). `segment_mass.csv` ghi tỷ lệ attention dồn vào từng phần.
Dùng checkpoint `PREDICT_CKPT` (cell 2). Đặt `ATTN_IDS` để chọn câu hỏi cụ thể; để trống thì lấy ngẫu nhiên `ATTN_N` câu (cùng seed cho mọi cấu hình). Hình được upload vào `attention/` (hoặc `attention/checkpoint-N/`) trên repo HF.

In [ ]:
import os, glob
from IPython.display import Image, display
!git checkout -q --detach {CODE_SHA}
ATTN_SPLIT = "test"
ATTN_N = 8
ATTN_IDS = []

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, ATTN_DIR = CKPT_INFO["model_dir"], os.path.join(CKPT_INFO["pred_dir"], "attention")
ATTN_REPO_PATH = CKPT_INFO["attn_path_in_repo"]
ids_arg = ("--ids " + " ".join(map(str, ATTN_IDS))) if ATTN_IDS else f"--n {ATTN_N}"
gt_arg = f"--gt eval_gt/{DATASET}_test_gt.json" if os.path.exists(f"eval_gt/{DATASET}_test_gt.json") else ""
!python scripts/visualize_attention.py --ckpt_dir {MODEL_DIR} --dataset {DATASET} --split {ATTN_SPLIT} {ids_arg} {gt_arg} --out_dir {ATTN_DIR}

pngs = sorted(glob.glob(os.path.join(ATTN_DIR, "*.png")))
for p in pngs:
    display(Image(filename=p))
if os.environ.get("HF_TOKEN") and HF_REPO and pngs:
    names = " ".join(os.path.basename(p) for p in pngs + [os.path.join(ATTN_DIR, "segment_mass.csv")])
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {ATTN_DIR} --path_in_repo {ATTN_REPO_PATH} --files {names}